# Energy Indicators, Basic Tutorial

Runs `energy_indicators` standalone (outside the ClimateDT workflow),
using real Climate DT data for Tico (Villar de los Navarros, Zaragoza,
Spain, onshore wind farm) and its real regional neighbours: Zaragoza
(city, demand) and Fuendetodos (solar farm, covered in the Advanced
Tutorial). January 2035, SSP3-7.0 scenario, retrieved via
`data_retrieval_earthkit.ipynb`. These files are already provided in
the repository, this notebook doesn't require DESP credentials itself.

It covers a world map of monthly-mean temperature, a European context
map showing all four sites used across the Basic and Advanced
tutorials (including Moray East, Scotland, covered in the Advanced
Tutorial's wind comparison), Capacity Factor and Wind Power Density
at Tico, and Cooling and Heating Degree Days at Zaragoza.

Tico's real installed turbines (Siemens-Gamesa SG 5.0-132, SG
3.4-132) are approximated to be IEC Class III, not a genuine match.

This uses the lower-level indicator functions (`capacity_factor`,
`wind_power_density`, etc.) directly, since the retrieved data isn't
in the file layout the `run_*` wrappers expect. For the file-based
`run_*` pattern, see
`docs/source/user_guide/without_workflow/first_dummy_run.rst`.

All data below is at low resolution (HEALPix H128, ~50 km
native cells), so each site's data is the nearest available cell, not
its exact coordinate.

## Setup

Requires these files from `data_retrieval_earthkit.ipynb`:
`data/tico_wind_203501.nc`, `data/zaragoza_temp_203501.nc`,
`data/global_2t_monthly_203501.nc`, `data/europe_2t_203501.nc`.

In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy.interpolate import griddata
from IPython.display import Image

from energy_indicators.core import wind_speed, convert_temperature
from energy_indicators.wind import capacity_factor, wind_power_density
from energy_indicators.demand import cooling_degree_days, heating_degree_days
from energy_indicators.plot import plot_map, define_colormaps

data_path = "data/"
out_path = "./tutorial_output/basic/"
os.makedirs(out_path, exist_ok=True)

colormaps = define_colormaps()

SITE_NAME = "Tico"
IEC_CLASS = "III"

sites = {
    "Tico": {"lat": 41.159, "lon": -1.044, "type": "onshore wind"},
    "Zaragoza": {"lat": 41.656, "lon": -0.877, "type": "city (demand)"},
    "Fuendetodos": {"lat": 41.200, "lon": -1.283, "type": "solar"},
    "Moray East": {"lat": 58.167, "lon": -2.699, "type": "offshore wind (Advanced)"},
}


## What's in the package

`wind.py` covers wind indicators: Wind Speed Anomalies, Wind Power
Density, Capacity Factor (and its histograms), high/low Wind Events,
Wind Producing Regimes, and Annual Energy Production. `demand.py`
covers Cooling and Heating Degree Days. `solar.py` covers PV
Potential.

Every function has a full docstring, with the exact input/output
shapes and units:

In [ ]:
help(capacity_factor)


Every retrieved point time series has shape `(time,)` (Tico's wind
also carries a leftover, size-1 `level` dimension). Every
`energy_indicators` function expects gridded input, `(time, lat,
lon)`, even for a single point. This helper wraps a point series into
that shape, attaching the site's actual retrieved coordinates.

In [ ]:
def as_grid(da, lat, lon):
    da = da.squeeze()
    return da.expand_dims({"lat": [lat], "lon": [lon]}).transpose("time", "lat", "lon")


def regrid_with_fallback(lons, lats, values, lon_grid, lat_grid):
    values_grid = griddata(points=(lons, lats), values=values, xi=(lon_grid, lat_grid), method="linear")
    nan_mask = np.isnan(values_grid)
    if nan_mask.any():
        values_nearest = griddata(points=(lons, lats), values=values, xi=(lon_grid, lat_grid), method="nearest")
        values_grid[nan_mask] = values_nearest[nan_mask]
    return values_grid


## Global temperature

A genuine global field, retrieved as native HEALPix, regridded to a
regular grid for plotting.

In [ ]:
global_ds = xr.open_dataset(data_path + "global_2t_monthly_203501.nc")
lons = global_ds["lon"].values
lats = global_ds["lat"].values
values = global_ds["2t"].values.ravel()

grid_deg = 1.0
lon_new = np.arange(-180, 180, grid_deg)
lat_new = np.arange(-90, 90, grid_deg)
lon_grid, lat_grid = np.meshgrid(lon_new, lat_new)
values_grid = griddata(points=(lons, lats), values=values, xi=(lon_grid, lat_grid), method="nearest")

da_global = xr.DataArray(values_grid, coords={"lat": lat_new, "lon": lon_new}, dims=("lat", "lon"))

plot_map(
    data=da_global,
    xmin=float(np.nanmin(values_grid)), xmax=float(np.nanmax(values_grid)),
    colormap=colormaps["cmap_cmocean"],
    label="2 m Temperature (K)",
    title="Global mean 2 m Temperature, 2035-01 (monthly mean, SSP3-7.0)",
    output_file=out_path + "global_2t.png",
)
Image(out_path + "global_2t.png")


## European context map

Tico (wind), Zaragoza (demand), and Fuendetodos (solar) are all in
Zaragoza province, within about 50 km of each other, a real regional
cluster, not arbitrary points. Moray East (Scotland), the Advanced
Tutorial's second wind site, is marked too, for orientation.

`plot_map` doesn't currently support marker overlays, so this map is
built directly with matplotlib and cartopy instead.

In [ ]:
def plot_context_map(lons_full, lats_full, values_full, bbox, sites_to_show,
                      label_offsets, title, output_file, grid_deg=0.25):
    n, w, s, e = bbox
    mask = (lats_full <= n) & (lats_full >= s) & (lons_full >= w) & (lons_full <= e)

    lon_new = np.arange(w, e, grid_deg)
    lat_new = np.arange(s, n, grid_deg)
    lon_grid, lat_grid = np.meshgrid(lon_new, lat_new)
    values_grid = regrid_with_fallback(lons_full[mask], lats_full[mask], values_full[mask], lon_grid, lat_grid)

    fig, ax = plt.subplots(figsize=(12, 10), subplot_kw={"projection": ccrs.PlateCarree()})
    cs = ax.pcolormesh(lon_grid, lat_grid, values_grid, transform=ccrs.PlateCarree(),
                        cmap=colormaps["cmap_cmocean"], vmin=np.nanmin(values_grid), vmax=np.nanmax(values_grid))
    ax.add_feature(cfeature.LAND, edgecolor="black", facecolor="none")
    ax.add_feature(cfeature.COASTLINE, linewidth=0.75)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, linestyle="--")
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)
    gl.top_labels = False
    gl.right_labels = False

    for name in sites_to_show:
        site = sites[name]
        ax.plot(site["lon"], site["lat"], marker="o", color="black", markersize=8,
                 transform=ccrs.PlateCarree())
        ax.annotate(
            f"{name}\n({site['type']})",
            xy=(site["lon"], site["lat"]),
            xycoords=ccrs.PlateCarree()._as_mpl_transform(ax),
            xytext=label_offsets[name], textcoords="offset points",
            fontsize=9, ha="center",
            arrowprops=dict(arrowstyle="->", color="black", lw=1),
            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="black", alpha=0.85),
        )

    plt.colorbar(cs, ax=ax, label="2 m Temperature (K)", orientation="horizontal", pad=0.05, shrink=0.7)
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(output_file, dpi=150, bbox_inches="tight")
    plt.show()


europe_ds = xr.open_dataset(data_path + "europe_2t_203501.nc")
lons = europe_ds["lon"].values
lats = europe_ds["lat"].values
values = europe_ds["2t"].values.ravel()

plot_context_map(
    lons, lats, values,
    bbox=(72, -15, 35, 32),
    sites_to_show=["Tico", "Zaragoza", "Fuendetodos", "Moray East"],
    label_offsets={
        "Tico": (-60, -35), "Zaragoza": (50, 20),
        "Fuendetodos": (-70, 20), "Moray East": (30, 20),
    },
    title="Europe, 2 m Temperature context (2035-01)",
    output_file=out_path + "europe_context_marked.png",
)


## Wind Speed, Capacity Factor, Wind Power Density

`u`/`v` at 100 m, retrieved as a point time series for the full
month.

In [ ]:
wind = xr.open_dataset(data_path + "tico_wind_203501.nc")
TICO_LAT, TICO_LON = float(wind["lat"]), float(wind["lon"])

u = as_grid(wind["u"], TICO_LAT, TICO_LON)
v = as_grid(wind["v"], TICO_LAT, TICO_LON)

ws = wind_speed(u, v)
ws


In [ ]:
cf = capacity_factor(ws, iec_class=IEC_CLASS)
cf


In [ ]:
cf_series = cf[f"cf_{IEC_CLASS.lower()}"].squeeze()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(cf_series["time"], cf_series * 100, color="green")
ax.set_xlabel("Time"); ax.set_ylabel("Capacity Factor (%)")
ax.set_title(f"{SITE_NAME}, Capacity Factor (Class {IEC_CLASS}), 2035-01")
ax.grid(True)
plt.xticks(rotation=30); plt.tight_layout()
plt.savefig(out_path + "cf_timeseries.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
wpd_series = wind_power_density(ws).squeeze()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(wpd_series["time"], wpd_series, color="darkorange")
ax.set_xlabel("Time"); ax.set_ylabel("Wind Power Density (W/m²)")
ax.set_title(f"{SITE_NAME}, Wind Power Density, 2035-01")
ax.grid(True)
plt.xticks(rotation=30); plt.tight_layout()
plt.savefig(out_path + "wpd_timeseries.png", dpi=150, bbox_inches="tight")
plt.show()


## Cooling and Heating Degree Days, Zaragoza

`cooling_degree_days`/`heating_degree_days` need daily mean/max/min
temperature in Celsius. Climate DT data arrives in Kelvin, so it must
be converted first (`base=22.0`/`15.5` are Celsius thresholds;
skipping this conversion silently produces nonsensical results, since
the functions have no way to detect the wrong unit).

In [ ]:
temp = xr.open_dataset(data_path + "zaragoza_temp_203501.nc")
ZARAGOZA_LAT, ZARAGOZA_LON = float(temp["lat"]), float(temp["lon"])

t2_k = temp["2t"]
t2_k.attrs["units"] = "K"
t2_c = convert_temperature(t2_k, unit="C")

tm_daily = as_grid(t2_c.resample(time="1D").mean(), ZARAGOZA_LAT, ZARAGOZA_LON)
tx_daily = as_grid(t2_c.resample(time="1D").max(), ZARAGOZA_LAT, ZARAGOZA_LON)
tn_daily = as_grid(t2_c.resample(time="1D").min(), ZARAGOZA_LAT, ZARAGOZA_LON)

cdd = cooling_degree_days(tm_daily, tx_daily, tn_daily, base=22.0)
hdd = heating_degree_days(tm_daily, tx_daily, tn_daily, base=15.5)
cdd


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(cdd["time"], cdd["cdd"].squeeze(), label="CDD", color="red")
ax.plot(hdd["time"], hdd["hdd"].squeeze(), label="HDD", color="blue")
ax.set_xlabel("Time"); ax.set_ylabel("Degree Days (°C)")
ax.set_title("Zaragoza, Cooling/Heating Degree Days, 2035-01")
ax.legend(); ax.grid(True)
plt.xticks(rotation=30); plt.tight_layout()
plt.savefig(out_path + "cdd_hdd_timeseries.png", dpi=150, bbox_inches="tight")
plt.show()


For the Tico vs Moray East wind comparison, masking, histograms,
Annual Energy Production, and Solar, see the Advanced Tutorial
(`tutorial_advanced.ipynb`).